# 05. 행렬 분해 (MF) — 배혜림 B1~B3

제안서 3.1. 위에서부터 순서대로 실행

| 셀 | 내용 | 대략 |
|---|---|---|
| 0~1 | 설정, 데이터 | 2분 |
| 2 | **하이퍼파라미터 탐색 (B2)** | 20분 |
| 3 | **MF 본 학습 + 예측값 저장** (지훈님 10/31 요청) | 5~10분 |
| 4 | 기준 모델 대비 (성공 기준 ①) | 1분 |
| 5 | 구성요소 분석 — B3 | 20분 |
| 6 | 콜드스타트 구간별 | 1분 |
| 7 | 랭킹 ①②③ | 5분 |
| 8 | 저장 + 단톡 공유 문구 | 1분 |


**넘어야 할 기준**

| | Video_Games | Toys_and_Games |
|---|---|---|
| 성공 기준 ①·④ RMSE | **1.247 미만** | **1.217 미만** |
| 성공 기준 ③ 인기도 HR@10 (프로토콜 ③) | **0.0248 초과** | **0.0038 초과** |

In [1]:
# ===== 0. 설정 =====
import os, sys, subprocess, time
try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = '/content/team10-recsys'
    if os.path.exists(REPO):
        subprocess.run(['git', '-C', REPO, 'pull', '-q'])
    else:
        subprocess.run(['git', 'clone', '-q',
                        'https://github.com/qhfl3644/team10-recsys.git', REPO])
    sys.path.insert(0, REPO)
except ImportError:
    REPO = os.path.abspath('..')
    sys.path.insert(0, REPO)

import numpy as np, pandas as pd
from config import DATASETS, PROC_DIR, PRED_DIR, RESULT_DIR, SEED
from src.eval import metrics as M
from src.models.baselines import BASELINES
from src.models.mf import MF

TARGET = {'Video_Games': dict(rmse=1.247, pop_hr3=0.0248),
          'Toys_and_Games': dict(rmse=1.217, pop_hr3=0.0038)}
DS = 'Video_Games'
print('준비 완료:', DS)

Mounted at /content/drive
준비 완료: Video_Games


## 1. 데이터

`train`으로 학습하고 `valid`로 조기 종료한다. `test`는 마지막에 한 번만 쓴다.

In [2]:

# ===== 1. 데이터 =====
t0 = time.time()
data = M.load_data(DS)
unseen_test = ~data.seen_mask(data.test)
print(f'{DS}: 사용자 {data.n_users:,} / 상품 {data.n_items:,}')
print(f'  train {len(data.train):,} / valid {len(data.valid):,} / test {len(data.test):,}')
print(f'  train 평점 3~4개 사용자 {(data.train_count <= 4).sum():,}명')
print(f'  train에 없는 상품이 정답인 test 행 {unseen_test.sum():,}개 ({unseen_test.mean()*100:.2f}%)')
print(f'  ({time.time()-t0:.0f}초)')

Video_Games: 사용자 94,762 / 상품 25,612
  train 625,062 / valid 94,762 / test 94,762
  train 평점 3~4개 사용자 48,624명
  train에 없는 상품이 정답인 test 행 345개 (0.36%)
  (6초)


## 2. 하이퍼파라미터 탐색 (B2)

잠재요인 개수 k, 학습률 lr, 정규화 강도 reg를 **valid RMSE로만** 비교해 고른다
(공통 규칙: test는 마지막 한 번).

1단계에서 학습률을 먼저 정하고, 2단계에서 그 학습률로 k × reg 격자를 돈다. 전체 27개 조합을 다 보는 격자 탐색이 아니라 2단계 12회 탐색이다.
탐색은 에폭을 줄여 돌리고, 고른 값으로 3번 셀에서 본 학습을 한다.

In [3]:
# ===== 2. 하이퍼파라미터 탐색 (B2) =====
t0 = time.time()
GRID_LR = [0.005, 0.01, 0.02]
GRID_K = [8, 16, 32]
GRID_REG = [0.05, 0.1, 0.2]
TUNE_EPOCHS = 25

def try_one(k, lr, reg, stage):
    m = MF(k=k, lr=lr, reg=reg, epochs=TUNE_EPOCHS, patience=3, seed=SEED, verbose=False)
    m.fit(data.train, data.n_users, data.n_items, valid=data.valid)
    print(f'  [{stage}] k={k:<3} lr={lr:<6} reg={reg:<5} valid RMSE {m.best_valid:.4f}'
          f'  ({time.time()-t0:.0f}초)')
    return dict(단계=stage, k=k, lr=lr, reg=reg, valid_rmse=m.best_valid, 최저에폭=m.best_epoch)

# 1단계: 학습률 (k, reg는 가운데 값으로 고정)
rows = [try_one(16, lr, 0.1, '1-lr') for lr in GRID_LR]
BEST_LR = float(min(rows, key=lambda r: r['valid_rmse'])['lr'])
print(f'\n→ 학습률 선택: {BEST_LR}\n')

# 2단계: 선택된 학습률에서 k x reg
rows += [try_one(k, BEST_LR, reg, '2-k·reg') for k in GRID_K for reg in GRID_REG]

tuning = pd.DataFrame(rows).sort_values('valid_rmse').reset_index(drop=True)
best = tuning.iloc[0]
BEST_K, BEST_REG, BEST_LR = int(best.k), float(best.reg), float(best.lr)
print()
print(tuning.round(4).to_string(index=False))
print(f'\n선택: k={BEST_K}, lr={BEST_LR}, reg={BEST_REG} (valid RMSE {best.valid_rmse:.4f})')

  [1-lr] k=16  lr=0.005  reg=0.1   valid RMSE 1.0983  (226초)
  [1-lr] k=16  lr=0.01   reg=0.1   valid RMSE 1.0906  (453초)
  [1-lr] k=16  lr=0.02   reg=0.1   valid RMSE 1.0914  (597초)

→ 학습률 선택: 0.01

  [2-k·reg] k=8   lr=0.01   reg=0.05  valid RMSE 1.0914  (803초)
  [2-k·reg] k=8   lr=0.01   reg=0.1   valid RMSE 1.0906  (1025초)
  [2-k·reg] k=8   lr=0.01   reg=0.2   valid RMSE 1.0909  (1247초)
  [2-k·reg] k=16  lr=0.01   reg=0.05  valid RMSE 1.0915  (1461초)
  [2-k·reg] k=16  lr=0.01   reg=0.1   valid RMSE 1.0906  (1686초)
  [2-k·reg] k=16  lr=0.01   reg=0.2   valid RMSE 1.0909  (1910초)
  [2-k·reg] k=32  lr=0.01   reg=0.05  valid RMSE 1.0924  (2139초)
  [2-k·reg] k=32  lr=0.01   reg=0.1   valid RMSE 1.0911  (2386초)
  [2-k·reg] k=32  lr=0.01   reg=0.2   valid RMSE 1.0910  (2622초)

     단계  k    lr  reg  valid_rmse  최저에폭
   1-lr 16 0.010 0.10      1.0906    22
2-k·reg 16 0.010 0.10      1.0906    22
2-k·reg  8 0.010 0.10      1.0906    22
2-k·reg 16 0.010 0.20      1.0909    24
2-k·reg  8 0.01

## 3. MF 본 학습 + 예측값 저장

편향 항 + L2 정규화 + SGD, 검증 오차 기준 조기 종료.

**train에 한 번도 안 나온 상품 처리**
CBF가 없어 MF 자체 값으로 채워서 올리고 (해당 상품은 잠재요인을 0으로 두어 예측이 정확히 전역평균 + 사용자편향이 된다),
해당 행 목록을 따로 저장해 CBF가 나오면 바로 바꿔 끼울 수 있게 한다.
지표 함수의 `scope='seen'`은 이 행들을 빼고 계산하므로 순수 MF 비교에는 영향이 없다.

In [4]:
# ===== 3. MF 본 학습 =====
t0 = time.time()
mf = MF(k=BEST_K, lr=BEST_LR, reg=BEST_REG, epochs=60, patience=5, seed=SEED)
mf.fit(data.train, data.n_users, data.n_items, valid=data.valid)
print(f'학습 완료 ({time.time()-t0:.0f}초)')

for split in ['valid', 'test']:
    df = getattr(data, split)
    pred = mf.predict(df['uidx'].values, df['iidx'].values)
    print(' 저장:', M.save_predictions(DS, 'mf', split, df, pred))

    # CBF 대체가 필요한 행 목록 (지훈님 CBF 나오면 교체)
    un = ~data.seen_mask(df)
    if un.any():
        p = f'{PRED_DIR}/{DS}/mf_unseen_{split}.parquet'
        df.loc[un, ['user_id', 'item_id', 'uidx', 'iidx']].to_parquet(p, index=False)
        print(f'   CBF 대체 대상 {int(un.sum()):,}행 → {os.path.basename(p)}')

y = data.test['rating'].values
p_test = mf.predict(data.test['uidx'].values, data.test['iidx'].values)
print(f'\ntest RMSE {M.rmse(y, p_test):.4f}  MAE {M.mae(y, p_test):.4f}')

    epoch   1  valid RMSE 1.1763
    epoch   2  valid RMSE 1.1579
    epoch   3  valid RMSE 1.1454
    epoch   5  valid RMSE 1.1274
    epoch  10  valid RMSE 1.1043
    epoch  15  valid RMSE 1.0950
    epoch  20  valid RMSE 1.0913
    epoch  25  valid RMSE 1.0908
    조기 종료: 27에폭 (최저 22에폭, valid RMSE 1.0906)
학습 완료 (258초)
 저장: /content/drive/MyDrive/Team10_RecSys/predictions/Video_Games/mf_valid.parquet
   CBF 대체 대상 163행 → mf_unseen_valid.parquet
 저장: /content/drive/MyDrive/Team10_RecSys/predictions/Video_Games/mf_test.parquet
   CBF 대체 대상 345행 → mf_unseen_test.parquet

test RMSE 1.1714  MAE 0.8313


## 4. 기준 모델 대비 (성공 기준 ①)

In [5]:
# ===== 4. 기준 모델 대비 =====
base = {k: cls().fit(data.train, data.n_users, data.n_items) for k, cls in BASELINES.items()}
preds = {k: m.predict(data.test['uidx'].values, data.test['iidx'].values) for k, m in base.items()}
preds['mf'] = p_test

rt = M.rating_table(preds, data.test, data)
print(rt[(rt.scope == 'all') & (rt.group == 'all')][['model', 'n', 'rmse', 'mae']].round(4).to_string(index=False))

c1 = M.criterion_rating_vs_baselines(p_test, {k: preds[k] for k in ['global_mean', 'user_mean', 'item_mean']},
                                     data.test, data, scope='all')
mine = M.rmse(y, p_test)
print(f"\n[성공 기준 ①] {'통과' if c1.attrs['passed'] else '실패'}"
      f"  (MF {mine:.4f} vs 목표 {TARGET[DS]['rmse']} 미만)")
print(c1[['baseline', 'rmse_model', 'rmse_baseline', 'ci_lo', 'ci_hi', 'passed']].round(4).to_string(index=False))

      model     n   rmse    mae
global_mean 94762 1.2949 1.0049
  user_mean 94762 1.2466 0.8267
  item_mean 94762 1.2677 0.9316
 popularity 94762 1.2677 0.9316
         mf 94762 1.1714 0.8313

[성공 기준 ①] 통과  (MF 1.1714 vs 목표 1.247 미만)
   baseline  rmse_model  rmse_baseline  ci_lo  ci_hi  passed
global_mean      1.1714         1.2949 0.1194 0.1274    True
  user_mean      1.1714         1.2466 0.0718 0.0787    True
  item_mean      1.1714         1.2677 0.0930 0.1001    True


## 5. 구성요소 분석 (B3)

In [6]:
# ===== 5. 구성요소 분석 =====
t0 = time.time()
abl = [dict(구성=f'전체 (k={BEST_K}, lr={BEST_LR}, reg={BEST_REG})', test_rmse=M.rmse(y, p_test),
            valid_rmse=mf.best_valid, 최저에폭=mf.best_epoch)]
for label, kw in [('bias만 (k=0)', dict(k=0, lr=BEST_LR, reg=BEST_REG)),
                  ('정규화 없음 (reg=0)', dict(k=BEST_K, lr=BEST_LR, reg=0.0)),
                  ('k=4', dict(k=4, lr=BEST_LR, reg=BEST_REG))]:
    m = MF(epochs=60, patience=5, seed=SEED, verbose=False, **kw)
    m.fit(data.train, data.n_users, data.n_items, valid=data.valid)
    pp = m.predict(data.test['uidx'].values, data.test['iidx'].values)
    abl.append(dict(구성=label, test_rmse=M.rmse(y, pp), valid_rmse=m.best_valid, 최저에폭=m.best_epoch))
    print(f'  {label:22s} test RMSE {abl[-1]["test_rmse"]:.4f}  ({time.time()-t0:.0f}초)')

ablation = pd.DataFrame(abl)
ablation['전체대비'] = (ablation.test_rmse - ablation.test_rmse[0]).round(4)
print()
print(ablation.round(4).to_string(index=False))

  bias만 (k=0)            test RMSE 1.1712  (37초)
  정규화 없음 (reg=0)         test RMSE 1.1775  (230초)
  k=4                    test RMSE 1.1712  (465초)

                         구성  test_rmse  valid_rmse  최저에폭    전체대비
전체 (k=16, lr=0.01, reg=0.1)     1.1714      1.0906    22  0.0000
                bias만 (k=0)     1.1712      1.0906    22 -0.0002
             정규화 없음 (reg=0)     1.1775      1.0981    16  0.0061
                        k=4     1.1712      1.0907    22 -0.0001


## 6. 콜드스타트 구간별 RMSE

성공 기준 ②는 하이브리드가 나와야 판정한다. 여기서는 MF 값만 확보해 둔다.

In [7]:
# ===== 6. 구간별 =====
seg = rt[(rt.scope == 'all') & (rt.group != 'all')].pivot_table(index='group', columns='model', values='rmse')
print(seg.round(4).to_string())

model        global_mean  item_mean      mf  popularity  user_mean
group                                                             
active(10+)       1.2812     1.2564  1.1331      1.2564     1.1737
cold(3-4)         1.3046     1.2749  1.1912      1.2749     1.2896
mid(5-9)          1.2860     1.2617  1.1574      1.2617     1.2107


## 7. 랭킹 평가 (①②③)

제안서 2.1절. ③이 판정 기준이고 ①②는 비교용이다.
프로토콜 간 비교는 `p3_users` 1만 명 공통 표본으로 한다 (EVAL_RULES).

In [8]:
# ===== 7. 랭킹 =====
t0 = time.time()
p3 = M.load_p3_users(DS)
tbl, hits = M.ranking_eval({**base, 'mf': mf}, data, 'test', users=p3, chunk=256)
piv = tbl.pivot_table(index='model', columns='protocol', values='HR10')
print(piv.round(4).to_string())
print(f"\n무작위 추측: ①② {10/100:.2f} / ③ {10/data.n_items:.5f}")
print(f"인기도 ③ {piv.loc['popularity', 'p3']:.4f}  (율이님 값 {TARGET[DS]['pop_hr3']})")
print(f'({time.time()-t0:.0f}초)')

protocol         p1      p2      p3
model                              
global_mean  0.0983  0.0983  0.0006
item_mean    0.0542  0.1157  0.0003
mf           0.1099  0.1097  0.0002
popularity   0.4919  0.1660  0.0248
user_mean    0.0983  0.0983  0.0006

무작위 추측: ①② 0.10 / ③ 0.00039
인기도 ③ 0.0248  (율이님 값 0.0248)
(16초)


## 8. 저장

In [13]:
# ===== 8. 저장 =====
OUT = f'{RESULT_DIR}/mf'; os.makedirs(OUT, exist_ok=True)
if 'dataset' not in rt.columns: rt.insert(0, 'dataset', DS)
rt.to_csv(f'{OUT}/{DS}_rating_test.csv', index=False)

if 'dataset' not in tbl.columns: tbl.insert(0, 'dataset', DS)
tbl.to_csv(f'{OUT}/{DS}_ranking_test.csv', index=False)

if 'dataset' not in ablation.columns: ablation.insert(0, 'dataset', DS)
ablation.to_csv(f'{OUT}/{DS}_ablation.csv', index=False)

if 'dataset' not in tuning.columns: tuning.insert(0, 'dataset', DS)
tuning.to_csv(f'{OUT}/{DS}_tuning.csv', index=False)

pd.DataFrame({'epoch': np.arange(1, len(mf.history)+1), 'valid_rmse': mf.history}).to_csv(
    f'{OUT}/{DS}_mf_history.csv', index=False)
M.save_json(dict(k=mf.k, lr=mf.lr, reg=mf.reg, grid=tuning.to_dict('records'), best_epoch=mf.best_epoch, best_valid=mf.best_valid,
                 test_rmse=M.rmse(y, p_test), test_mae=M.mae(y, p_test)), f'{OUT}/{DS}_mf_config.json')